In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

C:\Users\thewr\AppData\Local\Temp\ipykernel_6368\2565918290.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\thewr\git\LSTM_giroscopio_acelerometro.git\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [4]:
def equalize_dataframe_sizes(df1, df2):
    """Equalizes the sizes of two DataFrames by dropping rows from the larger one.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        tuple: A tuple containing the equalized DataFrames (df1, df2).

    Raises:
        ValueError: If both DataFrames have the same size.
    """

    if df1.shape[0] == df2.shape[0]:
        raise ValueError("Both DataFrames already have the same size.")

    larger_df, smaller_df = (df1, df2) if df1.shape[0] > df2.shape[0] else (df2, df1)

    # Drop rows from the larger DataFrame to match the smaller one
    larger_df = larger_df.iloc[: smaller_df.shape[0], :]

    return larger_df, smaller_df

# Example usage
#df1 = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': ['a', 'b', 'c', 'd', 'e']})
#df2 = pd.DataFrame({'A': [10, 20, 30], 'B': ['x', 'y', 'z']})

''' try:
    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())
    #print("Equalized DataFrames:")
    #print(df1_equal)
    #print(df2_equal)
except ValueError as e:
    print(e) '''


' try:\n    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())\n    #print("Equalized DataFrames:")\n    #print(df1_equal)\n    #print(df2_equal)\nexcept ValueError as e:\n    print(e) '

In [5]:
def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Create groups based on unique combinations of X and Y values
    groups_df1 = df1.groupby(['pessoa_gyro','atividade_gyro'])
    groups_df2 = df2.groupby(['pessoa', 'atividade'])

    # Iterate over groups from both DataFrames simultaneously
    for (x, y), group1 in groups_df1:
        #print(group1)
        #if (x,y) in [(1637, 'C'),(1638, 'H'),(1638, 'O'),(1640, 'C'),(1640, 'E')]:
        #    continue
        #else:
        group2 = groups_df2.get_group((x, y))
        '''  group1_has_nan = group1.isnull().sum().sum() > 0
        group2_has_nan = group2.isnull().sum().sum() > 0
        if group1_has_nan == True:
                print(group1.isnull().sum().sum())
                print (group1)
                break
        if group2_has_nan == True:
                print(group2.isnull().sum().sum())
                print (group2)
                break '''
        
        if group2 is not None:  # Check if corresponding group exists in df2
            if group1.shape[0] != group2.shape[0]:
                group1_equal, group2_equal = equalize_dataframe_sizes(group1.copy(), group2.copy())
                #print(f"+++++++++++++++++++++++++++++++++Imprimindo shape grupo 1 e 2 :{group1_equal.shape[0]} - {group2_equal.shape[0]}" )
                #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1_equal}" )
                #print("---------------------------------------------")
                #print(f"####################################Imprimindo dados grupo 2 :{group2_equal}" )
            #print(f"+++++++++++++++++++++++++++++++++Imprimindo shape grupo 1 e 2 :{group1_equal.shape[0]} - {group2_equal.shape[0]}" )
            #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1_equal}" )
            #print("---------------------------------------------")
            #print(f"####################################Imprimindo dados grupo 2 :{group2_equal}" ) 
            group1_equal = group1_equal.reset_index(drop=True) 
            group2_equal = group2_equal.reset_index(drop=True)  
            df_concat_group1_group2 = pd.concat([group1_equal, group2_equal], axis=1)
            #print(df_concat_group1_group2)
            df_concat_group1_group2 = df_concat_group1_group2.dropna()            
            #print(df_concat_group1_group2)
            has_nan = df_concat_group1_group2.isnull().sum().sum() > 0
            if has_nan == True:
                print (f"**************************** Imprimindo concat co nan: {df_concat_group1_group2}")
                break
            df_concat_group1_group2 = df_concat_group1_group2.reset_index(drop=True) 
            result_df = result_df.reset_index(drop=True)
            result_df = pd.concat([result_df, df_concat_group1_group2])

    return result_df

In [6]:
''' def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['pessoa'].unique():
        group_df1 = df1[df1['pessoa'] == x_val]
        for y_val in group_df1['atividade'].unique():
            group1 = group_df1[group_df1['atividade'] == y_val]
            group2 = df2[(df2['pessoa'] == x_val) & (df2['atividade'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])

    return result_df '''



' def custom_concat(df1, df2):\n    """\n    Concatenates two DataFrames with the same column names, considering specific conditions.\n\n    Args:\n        df1 (pd.DataFrame): The first DataFrame.\n        df2 (pd.DataFrame): The second DataFrame.\n\n    Returns:\n        pd.DataFrame: The concatenated DataFrame.\n    """\n\n    result_df = pd.DataFrame()\n\n    # Iterate over consecutive groups in each DataFrame\n    for x_val in df1[\'pessoa\'].unique():\n        group_df1 = df1[df1[\'pessoa\'] == x_val]\n        for y_val in group_df1[\'atividade\'].unique():\n            group1 = group_df1[group_df1[\'atividade\'] == y_val]\n            group2 = df2[(df2[\'pessoa\'] == x_val) & (df2[\'atividade\'] == y_val)]\n            if not group2.empty:  # Check if corresponding group exists in df2\n                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])\n\n    return result_df '

In [7]:
''' # Sample DataFrames
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A', 'B', 'B', 'B', 'B']})

def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['X'].unique():
        group_df1 = df1[df1['X'] == x_val]
        for y_val in group_df1['Y'].unique():
            group1 = group_df1[group_df1['Y'] == y_val]
            group2 = df2[(df2['X'] == x_val) & (df2['Y'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group1, group2], keys=[1, 2], axis=1, ignore_index=False)])

    return result_df

# Concatenate the DataFrames
concatenated_df = custom_concat(df1.copy(), df2.copy()) 

print(concatenated_df) '''

' # Sample DataFrames\ndf1 = pd.DataFrame({\'X\': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],\n                   \'Y\': [\'A\', \'A\', \'B\', \'B\', \'C\', \'A\', \'A\', \'B\', \'B\']})\ndf2 = pd.DataFrame({\'X\': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],\n                   \'Y\': [\'A\', \'A\', \'B\', \'B\', \'A\', \'B\', \'B\', \'B\', \'B\']})\n\ndef custom_concat(df1, df2):\n    """\n    Concatenates two DataFrames with the same column names, considering specific conditions.\n\n    Args:\n        df1 (pd.DataFrame): The first DataFrame.\n        df2 (pd.DataFrame): The second DataFrame.\n\n    Returns:\n        pd.DataFrame: The concatenated DataFrame.\n    """\n\n    result_df = pd.DataFrame()\n\n    # Iterate over consecutive groups in each DataFrame\n    for x_val in df1[\'X\'].unique():\n        group_df1 = df1[df1[\'X\'] == x_val]\n        for y_val in group_df1[\'Y\'].unique():\n            group1 = group_df1[group_df1[\'Y\'] == y_val]\n            

In [8]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/phone/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/phone/accel"

In [9]:
df_csv_gyro = combine_csv_data(csv_gyro_dir)

In [10]:
df_csv_gyro = df_csv_gyro.replace({';':''}, regex=True)

In [11]:
df_csv_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_x','giroscópio_Y','giroscópio_Z']

In [12]:
df_csv_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,252207918580802,-0.853210,0.297226,0.8901825
1,1600,A,252207968934806,-0.875137,0.015472,0.16223145
2,1600,A,252208019288809,-0.720169,0.388489,-0.28401184
3,1600,A,252208069642813,-0.571640,1.227402,-0.2416687
4,1600,A,252208119996817,-0.380493,1.202835,-0.21313477


In [13]:
df_csv_gyro.tail()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3608630,1650,S,357733977635000,0.438889,0.171508,0.16937704
3608631,1650,S,357734017925000,0.444215,0.158724,0.19707392
3608632,1650,S,357734058325000,0.519849,0.037284,0.19600865
3608633,1650,S,357734098619000,0.545415,-0.086286,0.17896444
3608634,1650,S,357734138868000,0.471912,-0.201335,0.23222765


In [14]:
df_csv_gyro.shape

(3608635, 6)

In [15]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3608635 entries, 0 to 3608634
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 165.2+ MB


In [16]:
''' condition1 = (df_csv_gyro['pessoa_gyro']== 1637) & (df_csv_gyro['atividade_gyro']=='C')
condition2 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='O')
condition3 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='C')
condition4 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='E')
condition5 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='H') '''

" condition1 = (df_csv_gyro['pessoa_gyro']== 1637) & (df_csv_gyro['atividade_gyro']=='C')\ncondition2 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='O')\ncondition3 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='C')\ncondition4 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='E')\ncondition5 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='H') "

In [17]:
#df_csv_gyro_filtered = df_csv_gyro[~(condition1 | condition2 | condition3 | condition4 | condition5)]

In [18]:
#df_csv_gyro_filtered = df_csv_gyro

In [19]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3608635 entries, 0 to 3608634
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 165.2+ MB


In [20]:
#test = df_csv_gyro_filtered[(df_csv_gyro_filtered['pessoa_gyro']==1638) & (df_csv_gyro_filtered['atividade_gyro']=='H') ]

In [21]:
#print(len(test))

In [22]:
#test2 = df_csv_gyro_filtered[df_csv_gyro_filtered['pessoa_gyro']==1638]

In [23]:
#print(len(test2))

In [24]:
#df_csv_gyro_filtered.shape

In [25]:
#df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [26]:
#df_csv_gyro.head()

In [27]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [28]:
df_csv_accel = df_csv_accel.replace({';':''}, regex=True)

In [29]:
df_csv_accel.shape

(4804403, 6)

In [30]:
df_csv_accel.columns = ['pessoa','atividade','tempo_acelerômetro','acelerômetro_x','acelerômetro_Y','acelerômetro_Z']

In [31]:
df_csv_accel.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
0,1600,A,252207666810782,-0.364761,8.793503,1.0550842
1,1600,A,252207717164786,-0.879730,9.768784,1.0169983
2,1600,A,252207767518790,2.001495,11.109070,2.619156
3,1600,A,252207817872794,0.450623,12.651642,0.18455505
4,1600,A,252207868226798,-2.164352,13.928436,-4.4224854


In [32]:
df_csv_accel.tail()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
4804398,1650,S,357734078421000,-8.234077,-4.275110,-1.4394068
4804399,1650,S,357734098619000,-8.080796,-4.418811,-1.4154567
4804400,1650,S,357734118715000,-7.994575,-4.457131,-1.4418018
4804401,1650,S,357734138868000,-8.040081,-4.366121,-1.4968873
4804402,1650,S,357734158964000,-8.164622,-4.179309,-1.5304176


In [33]:
#df_csv_accel.set_index(["pessoa_acelerômetro","atividade_acelerômetro","tempo_acelerômetro"],inplace=True)

In [34]:
#df_csv_accel.head()

In [35]:
''' condition_accel1 = (df_csv_accel['pessoa']== 1637) & (df_csv_accel['atividade']=='C')
condition_accel2 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='O')
condition_accel3 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='C')
condition_accel4 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='E')
condition_accel5 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='H') '''

" condition_accel1 = (df_csv_accel['pessoa']== 1637) & (df_csv_accel['atividade']=='C')\ncondition_accel2 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='O')\ncondition_accel3 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='C')\ncondition_accel4 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='E')\ncondition_accel5 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='H') "

In [36]:
#df_csv_accel_filtered = df_csv_accel[~(condition_accel1 | condition_accel2 | condition_accel3 | condition_accel4 | condition_accel5)]

In [37]:
#df_csv_accel_filtered = df_csv_accel

In [38]:
df_csv_accel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4804403 entries, 0 to 4804402
Data columns (total 6 columns):
 #   Column              Dtype  
---  ------              -----  
 0   pessoa              int64  
 1   atividade           object 
 2   tempo_acelerômetro  int64  
 3   acelerômetro_x      float64
 4   acelerômetro_Y      float64
 5   acelerômetro_Z      object 
dtypes: float64(2), int64(2), object(2)
memory usage: 219.9+ MB


In [39]:
#test3 = df_csv_accel_filtered[(df_csv_accel_filtered['pessoa']==1638) & (df_csv_accel_filtered['atividade']=='H') ]

In [40]:
#print(len(test3))

In [41]:
#test3.head()

In [42]:
#test4 = df_csv_accel_filtered[df_csv_accel_filtered['pessoa']==1638 ]

In [43]:
#print(len(test4))

In [44]:
#df_csv_gyro_filtered.shape

In [45]:
# Select rows with matching indices (inner join)
''' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1) '''

' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)\ndf1_filtered = df_csv_accel.loc[common_indices]\ndf2_filtered = df_csv_gyro.loc[common_indices]\n\n# Concatenate horizontally\nresult = pd.concat([df1_filtered, df2_filtered], axis=1) '

In [46]:
result_concatenation = custom_concat(df_csv_gyro.copy(),df_csv_accel.copy())

In [47]:
result_concatenation.shape

(3596093, 12)

In [48]:
result_concatenation.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,252207666810782,-0.364761,8.793503,1.0550842,1600,A,252207918580802,-0.853210,0.297226,0.8901825
1,1600,A,252207717164786,-0.879730,9.768784,1.0169983,1600,A,252207968934806,-0.875137,0.015472,0.16223145
2,1600,A,252207767518790,2.001495,11.109070,2.619156,1600,A,252208019288809,-0.720169,0.388489,-0.28401184
3,1600,A,252207817872794,0.450623,12.651642,0.18455505,1600,A,252208069642813,-0.571640,1.227402,-0.2416687
4,1600,A,252207868226798,-2.164352,13.928436,-4.4224854,1600,A,252208119996817,-0.380493,1.202835,-0.21313477


In [49]:
result_concatenation = result_concatenation.reset_index(drop=True)

In [50]:
result_concatenation.tail()


,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3596088,1650,S,357644019480000,-7.419771,-6.492898,-0.119750984,1650,S,357733977635000,0.438889,0.171508,0.16937704
3596089,1650,S,357644039687000,-7.644903,-7.391031,-0.64186525,1650,S,357734017925000,0.444215,0.158724,0.19707392
3596090,1650,S,357644059783000,-7.822134,-7.759863,-0.9915381,1650,S,357734058325000,0.519849,0.037284,0.19600865
3596091,1650,S,357644079967000,-7.125184,-6.569539,-1.1496094,1650,S,357734098619000,0.545415,-0.086286,0.17896444
3596092,1650,S,357644100063000,-6.159990,-5.316944,-1.1352394,1650,S,357734138868000,0.471912,-0.201335,0.23222765


In [51]:
test_diff_atividade = result_concatenation[result_concatenation['atividade']!=result_concatenation['atividade_gyro']]

In [52]:
print(test_diff_atividade.shape[0])

0


In [53]:
test_diff_pessoa = result_concatenation[result_concatenation['pessoa']!=result_concatenation['pessoa_gyro']]

In [54]:
print(len(test_diff_pessoa))

0


In [55]:
result_concatenation.columns

Index(['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'pessoa_gyro', 'atividade_gyro',
       'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'],
      dtype='object')

In [56]:
result_concatenation = result_concatenation[['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']]

In [57]:
result_concatenation.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,252207666810782,-0.364761,8.793503,1.0550842,252207918580802,-0.853210,0.297226,0.8901825
1,1600,A,252207717164786,-0.879730,9.768784,1.0169983,252207968934806,-0.875137,0.015472,0.16223145
2,1600,A,252207767518790,2.001495,11.109070,2.619156,252208019288809,-0.720169,0.388489,-0.28401184
3,1600,A,252207817872794,0.450623,12.651642,0.18455505,252208069642813,-0.571640,1.227402,-0.2416687
4,1600,A,252207868226798,-2.164352,13.928436,-4.4224854,252208119996817,-0.380493,1.202835,-0.21313477


In [58]:
result_concatenation = result_concatenation.rename(columns={'pessoa':'pessoa_x'})

In [59]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,252207666810782,-0.364761,8.793503,1.0550842,252207918580802,-0.853210,0.297226,0.8901825
1,1600,A,252207717164786,-0.879730,9.768784,1.0169983,252207968934806,-0.875137,0.015472,0.16223145
2,1600,A,252207767518790,2.001495,11.109070,2.619156,252208019288809,-0.720169,0.388489,-0.28401184
3,1600,A,252207817872794,0.450623,12.651642,0.18455505,252208069642813,-0.571640,1.227402,-0.2416687
4,1600,A,252207868226798,-2.164352,13.928436,-4.4224854,252208119996817,-0.380493,1.202835,-0.21313477


In [60]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [61]:
result_concatenation["atividade"] = result_concatenation["atividade"].replace(to_replace=mapping_dict)

In [62]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,252207666810782,-0.364761,8.793503,1.0550842,252207918580802,-0.853210,0.297226,0.8901825
1,1600,0,252207717164786,-0.879730,9.768784,1.0169983,252207968934806,-0.875137,0.015472,0.16223145
2,1600,0,252207767518790,2.001495,11.109070,2.619156,252208019288809,-0.720169,0.388489,-0.28401184
3,1600,0,252207817872794,0.450623,12.651642,0.18455505,252208069642813,-0.571640,1.227402,-0.2416687
4,1600,0,252207868226798,-2.164352,13.928436,-4.4224854,252208119996817,-0.380493,1.202835,-0.21313477


In [63]:
result_concatenation.tail(50)

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3596043,1650,17,357643112628000,-9.019644,-4.404441,-0.9532178,357732164299000,-0.400539,0.631702,0.29614353
3596044,1650,17,357643132724000,-8.497530,-4.255950,-0.7161109,357732204609000,-0.633832,0.159790,0.19813919
3596045,1650,17,357643152935000,-8.650811,-4.057163,-0.70174074,357732245045000,-0.693487,0.084156,0.20240025
3596046,1650,17,357643173031000,-8.959768,-4.169729,-0.48379397,357732285366000,-0.555003,0.166181,0.21837921
3596047,1650,17,357643193247000,-8.727451,-4.354145,-0.031135255,357732325601000,-0.387756,0.180030,0.20985709
3596048,1650,17,357643213343000,-8.444840,-4.397256,0.33769777,357732365904000,-0.227967,0.170442,0.17576863
3596049,1650,17,357643233519000,-8.073611,-4.430786,0.7041358,357732406269000,-0.124636,0.121440,0.165116
3596050,1650,17,357643253615000,-7.529942,-4.509822,0.9987232,357732446695000,-0.155529,0.020240,0.112918034
3596051,1650,17,357643273682000,-7.474856,-4.555327,1.1783496,357732487017000,-0.291882,0.011718,0.08948222
3596052,1650,17,357643293778000,-7.829319,-4.751719,1.2957057,357732527233000,-0.413323,0.052198,0.101200126


In [64]:
result_concatenation.to_csv('activity_recognition_data_phone_concat_by_comparacao_grupos.csv', index=False)